# The Lazy Book Report

Your professor has assigned a book report on "The Red-Headed League" by Arthur Conan Doyle. 

You haven't read the book. And out of stubbornness, you won't.

But you *have* learned NLP. Let's use it to answer the professor's questions without reading.

## Setup

First, let's fetch the text from Project Gutenberg and prepare it for analysis.

In [14]:
# Fetch and prepare text - RUN THIS CELL FIRST
import os
import urllib.request
import re

os.makedirs("output", exist_ok=True)

url = 'https://www.gutenberg.org/files/1661/1661-0.txt'
req = urllib.request.Request(url, headers={'User-Agent': 'Python-urllib'})
with urllib.request.urlopen(req, timeout=30) as resp:
    text = resp.read().decode('utf-8')

# Strip Gutenberg boilerplate
text = text.split('*** START OF')[1].split('***')[1]
text = text.split('*** END OF')[0]

# Extract "The Red-Headed League" story (it's the second story in the collection)
matches = list(re.finditer(r'THE RED-HEADED LEAGUE', text, re.IGNORECASE))
story_start = matches[1].end()
story_text = text[story_start:]
story_end = re.search(r'\n\s*III\.\s*\n', story_text)
story_text = story_text[:story_end.start()] if story_end else story_text

# Split into 3 sections by word count
words = story_text.split()[:4000]
section_size = len(words) // 3
sections = [
    ' '.join(words[:section_size]),
    ' '.join(words[section_size:2*section_size]),
    ' '.join(words[2*section_size:])
]

print(f"Story loaded: {len(words)} words in {len(sections)} sections")
print(f"Section sizes: {[len(s.split()) for s in sections]}")

Story loaded: 4000 words in 3 sections
Section sizes: [1333, 1333, 1334]


## Professor's Questions

Your professor wants you to answer 5 questions about the story. Let's use NLP to find the answers.

---

## Question 1: Writing Style

> "This text is from the 1890s. What makes it different from modern writing?"

**NLP Method:** Use preprocessing to compute text statistics. Tokenize the text and calculate:
- Vocabulary richness (unique words / total words)
- Average sentence length
- Average word length

**Hint:** Formal, literary writing typically shows higher vocabulary richness and longer sentences than modern casual text.

In [15]:
# Your code here: compute text statistics
# You'll need: import string, import re
# - Tokenize: remove punctuation, lowercase
# - Sentences: split on sentence-ending punctuation
# Calculate vocab_richness, avg_sentence_length, avg_word_length
import string

# deal with common abbreviations
abbreviations = ['Mr.', 'Mrs.', 'Ms.', 'Dr.', 'St.', 'etc.']
text_flag = story_text

for a in abbreviations:
    text_flag = text_flag.replace(a, a.replace('.', 'PROT'))

# split sentences apart
sentence = re.split(r'[.!?]+', text_flag)
sentence_vector = [s.replace('PROT', '.').strip() for s in sentence if s.strip()]

# remove punctuation and lowercase
tokenized = [s.lower() for s in sentence_vector]
tokenized_vector = [s.translate(str.maketrans('', '', string.punctuation)) for s in tokenized]

# find words
words = [word for s in tokenized_vector for word in re.findall(r'[a-z]+', s)]

# calculate text statistics

vocab_richness = len(set(words)) / len(words)
avg_sentence_length = len(words) / len(sentence_vector)
avg_word_length = sum([len(word) for word in words]) / len(words)

print(f"Vocabulary Richness: {vocab_richness:.4f}")
print(f"Average Sentence Length: {avg_sentence_length:.4f} words")
print(f"Average Word Length: {avg_word_length:.4f} characters")

Vocabulary Richness: 0.0800
Average Sentence Length: 15.6179 words
Average Word Length: 4.1002 characters


---

## Question 2: Main Characters

> "Who are the main characters in this story?"

**NLP Method:** Use Named Entity Recognition (NER) to extract PERSON entities.

**Hint:** Use spaCy's `en_core_web_sm` model. Process the text and filter entities where `ent.label_ == 'PERSON'`. Count how often each name appears.

In [16]:
# Your code here: extract PERSON entities using spaCy NER
# You'll need: import spacy, nlp = spacy.load("en_core_web_sm")

# When done, save your findings:
# with open("output/characters.txt", "w") as f:
#     for name in your_character_list:
#         f.write(f"{name}\n")
import spacy

nlp = spacy.load("en_core_web_sm")
doc = nlp(story_text)

# character names
characters = [ent.text for ent in doc.ents if ent.label_ == 'PERSON']

# now I will count how often each name appears 

character_counts = {}
for i in set(characters):
    character_counts[i] = characters.count(i)

print(character_counts)

# Saving my findings

with open("output/characters.txt", "w") as f:
    for name, counts in character_counts.items():
        f.write(f"{name}: {counts}\n")

{'Patience Moran': 1, 'James McCarthy': 4, 'Stoper': 5, 'John\r\nTurner': 1, 'Joseph': 1, 'Openshaw’s': 1, 'Upper Swandam Lane': 1, 'John\r\nCobb': 1, 'James Ryder': 2, 'Catherine\r\nCusack': 1, 'Saxon': 1, 'Jabez Wilson': 8, 'Calhoun': 1, 'James': 6, 'Near Lee': 1, 'Bill': 1, 'Mary Holder': 1, 'jewel-case': 1, 'Edward Street': 1, 'Clark Russell’s': 1, 'tawny': 1, 'Holder': 12, 'S. H. for J.\r\nO.”': 1, 'Charles McCarthy': 1, 'Neville St. Clair': 8, 'Francis H. Moulton': 1, 'James Windibank': 5, 'Lady St.\r\nSimon': 1, 'Neville St.': 1, 'Peter\r\nJones': 1, 'Henry Bakers': 1, 'Willows': 1, 'Henry Baker': 9, 'Stark': 2, 'Ross': 6, 'John': 3, 'Roylott’s': 2, 'Sholtos': 1, 'George Sand': 1, 'Hugh Boone': 3, 'Jack': 1, 'Jump': 2, 'Freebody': 1, 'Bristol': 1, 'Miss Stoner': 11, 'Dundee': 1, 'John Openshaw': 4, 'Gustave Flaubert': 1, 'L’homme c’est': 1, 'Turner': 10, 'Eton': 1, 'Grimesby\r\nRoylott': 1, 'Westaway’s': 1, 'Hereford': 1, 'gras': 1, 'Elias Whitney': 1, 'Fairbank': 2, 'George Bur

---

## Question 3: Story Locations

> "Where does the story take place?"

**NLP Method:** Use Named Entity Recognition (NER) to extract location entities (GPE and LOC).

**Hint:** Filter entities where `ent.label_` is 'GPE' (geopolitical entity) or 'LOC' (location).

In [17]:
# Your code here: extract GPE and LOC entities using spaCy NER

# When done, save your findings:
# with open("output/locations.txt", "w") as f:
#     for place in your_locations_list:
#         f.write(f"{place}\n")

location_entities = set([ent.text for ent in doc.ents if ent.label_ in ['GPE', 'LOC']])

# save findings
with open("output/locations.txt", "w") as f:
    for place in location_entities:
        f.write(f"{place}\n")

---

## Question 4: Wilson's Business

> "What is Wilson's business?"

**NLP Method:** Use TF-IDF similarity to find which section discusses Wilson's business.

**Hint:** Create a TF-IDF vectorizer, fit it on the 3 sections, then transform your query using the same vectorizer (`.transform()`, not `.fit_transform()` - you want to use the vocabulary learned from the sections). Find which section has the highest cosine similarity and read it to find the answer.

In [18]:
# Your code here: use TF-IDF similarity to find the relevant section
# You'll need: from sklearn.feature_extraction.text import TfidfVectorizer
#              from sklearn.metrics.pairwise import cosine_similarity

# When done, save your findings:
# with open("output/business.txt", "w") as f:
#     f.write("Wilson's business is: ...")

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# I'll set up the vectorizer
vectorizer = TfidfVectorizer(stop_words='english')
X = vectorizer.fit_transform(sections)

# Looking for Wilson's business
query = "Wilson's business"
query_vec = vectorizer.transform([query])
similarity_scores = cosine_similarity(query_vec, X).flatten()
print(similarity_scores)

# Now we extract highest similarity scores
best_similarity = similarity_scores.argmax()
best_section = sections[best_similarity]
sentences_similarity = re.findall(r'[^.!?]+[.!?]', best_section)

# now we find the sentence that mentions Wilson's business
for i in sentences_similarity:
    i = i.strip()
    if "business" in i.lower():
        print(i)
        break

# Now I will save my findings
with open("output/business.txt", "w") as f:
    f.write(f"Wilson's business is: a small pawnbroker’s business at Coburg Square, near the City.")

[0.09598401 0.18361727 0.15229363]
Sherlock Holmes,” said Jabez Wilson, mopping his forehead; “I have a small pawnbroker’s business at Coburg Square, near the City.


---

## Question 5: Wilson's Work Routine

> "What is Wilson's daily work routine for the League?"

**NLP Method:** Use TF-IDF similarity to find which section discusses Wilson's work routine.

**Hint:** Similar to Question 4 - use TF-IDF to find the section that best matches your query about work routine. The answer includes what Wilson had to do and what eventually happened.

In [19]:
# Your code here: use TF-IDF similarity to find the relevant section

# When done, save your findings:
# with open("output/routine.txt", "w") as f:
#     f.write("Wilson's work routine: ...\n")
#     f.write("What happened: ...\n")

# setting up new vectorizer
new_query = "Wilson's work routine"
new_query_vec = vectorizer.transform([new_query])
new_similarity_scores = cosine_similarity(new_query_vec, X).flatten()
print(new_similarity_scores)

# Now we extract the highest similarity scores 
new_best_similarity = new_similarity_scores.argmax()
new_best_section = sections[new_best_similarity]
new_sentences_similarity = re.findall(r'[^.!?]+[.!?]', new_best_section)

# Now, we will decide on the words to search for in order to find Wilson's work routine
words_to_search = ["wilson's", "daily", "work", "routine", "league"]
print(f"Words to search for: {words_to_search}")

for j, k in enumerate(new_sentences_similarity):
    k = k.strip().lower()
    if any(word in k for word in words_to_search):
        print(k, new_sentences_similarity[j + 1], '\n')

# Now I will save my findings
with open("output/routine.txt", "w") as f:
    f.write("Wilson's work routine: Is to copy out the _Encyclopædia Britannica_\n")
    f.write("What happened: the red-headed league is dissolved\n")

[0.0938429  0.06380381 0.12210878]
Words to search for: ["wilson's", 'daily', 'work', 'routine', 'league']
jabez wilson,’ said my assistant, ‘and he is willing to fill a vacancy in the league. ’ “‘And he is admirably suited for it,’ the other answered. 

’ “‘and the work? ’ “‘Is purely nominal. 

’ “‘and the work? ’ “‘Is to copy out the _Encyclopædia Britannica_. 

duncan ross was there to see that i got fairly to work.  He started me off upon the letter A, and then he left me; but he would drop in from time to time to see that all was right with me. 

holmes, and on saturday the manager came in and planked down four golden sovereigns for my week’s work.  It was the same next week, and the same the week after. 

i went to my work as usual at ten o’clock, but the door was shut and locked, with a little square of cardboard hammered on to the middle of the panel with a tack.  Here it is, and you can read for yourself. 

it read in this fashion: “the red-headed league is dissolved.  Octobe